## Setup

In [1]:
import os
import sys

# Add FleetPy to path if needed
fleetpy_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if fleetpy_path not in sys.path:
    sys.path.append(fleetpy_path)

# Import FleetPy modules
from src.misc.globals import *
import src.evaluation.tutorial_analysis as analysis
import src.misc.config as config
from src.misc.init_modules import load_simulation_environment
import pandas as pd

print("✅ FleetPy modules imported successfully!")

✅ FleetPy modules imported successfully!


In [ ]:
# Create a dictionary with the essential parameters using global variable names
config_params = {
    # Simulation environment
    # Use immediate decisions simulation environment
    G_SIM_ENV: 'ImmediateDecisionsSimulation',

    # No max decision time for immediate decisions
    G_AR_MAX_DEC_T: 0,
    # Basic request type for testing. Always accepts the operator's offer
    G_RQ_TYP1: 'BasicRequest',

    # Network and Demand
    G_NETWORK_NAME: 'example_network',             # Basic network for testing
    G_DEMAND_NAME: 'example_demand',               # Example demand pattern
    G_ZONE_SYSTEM_NAME: 'example_zones',           # Service area zones

    # Time Settings
    G_SIM_TIME_STEP: 30,                           # Update every 30 seconds
    G_SIM_START_TIME: 0,                           # Start at midnight
    G_SIM_END_TIME: 3600,                          # Run for 1 hour

    # Operational Settings
    G_NETWORK_TYPE: 'NetworkBasicWithStore',       # Basic network with store
    G_OP_MAX_WT: 300,                              # Max wait time of 5 minutes
    G_OP_MAX_DTF: 1.4,                             # Allow 40% detour
    # Constant boarding time of 30 seconds
    G_OP_CONST_BT: 30,
    G_NR_OPERATORS: 1,                             # Number of operators
    # Value of time function for vehicle routing control
    G_OP_VR_CTRL_F: 'func_key:distance_and_user_times_with_walk;vot:0.45',

    # Simulation Settings
    G_SLAVE_CPU: 1,                                # Use 1 CPU for slave processes
    "log_level": "info",                           # Set log level to INFO
    # Use a fixed random seed for reproducibility
    G_RANDOM_SEED: 0
}

# Convert to DataFrame for CSV format
config_df = pd.DataFrame(list(config_params.items()),
                         columns=['Parameter', 'Value'])

# Save constant config
config_dir = 'scenarios'
os.makedirs(config_dir, exist_ok=True)
constant_config_path = os.path.join(config_dir, 'custom_constant_config.csv')
config_df.to_csv(constant_config_path, index=False)

# Create a minimal scenario config (can override constant config values)
scenario_params = {
    G_STUDY_NAME: 'game_study',                    # Name of the study
    G_SCENARIO_NAME: 'example_pool_irsonly_sc_1',  # Scenario name
    G_RQ_FILE: "example_100.csv",                  # Example request file
    G_OP_FLEET: "default_vehtype:10",              # Size of the fleet
    # Operational module for pooling with IRS
    G_OP_MODULE: 'PoolingIRSOnly',
    G_OP_REPO_M: 'GameRepositioning'
}
scenario_df = pd.DataFrame([scenario_params])
scenario_path = os.path.join(config_dir, 'scenario_custom_config.csv')
scenario_df.to_csv(scenario_path, index=False)

print("✅ Custom configuration files created!")
print("\n📝 Current configuration:")
for param, value in config_params.items():
    print(f"  - {param:.<30} {value}")

✅ Custom configuration files created!

📝 Current configuration:
  - sim_env....................... ImmediateDecisionsSimulation
  - user_max_decision_time........ 0
  - rq_type....................... BasicRequest
  - network_name.................. example_network
  - demand_name................... example_demand
  - zone_system_name.............. example_zones
  - time_step..................... 30
  - start_time.................... 0
  - end_time...................... 3600
  - network_type.................. NetworkBasicWithStore
  - op_max_wait_time.............. 300
  - op_max_detour_time_factor..... 1.4
  - op_const_boarding_time........ 30
  - nr_mod_operators.............. 1
  - op_vr_control_func_dict....... func_key:distance_and_user_times_with_walk;vot:0.45
  - n_cpu_per_sim................. 1
  - log_level..................... info
  - random_seed................... 0


## Run Simulation

In [3]:
# Set up paths to our custom config files
scs_path = os.path.join(os.getcwd(), 'scenarios')
constant_config_file = os.path.join(scs_path, 'custom_constant_config.csv')
scenario_file = os.path.join(scs_path, 'scenario_custom_config.csv')

# Read configuration files
constant_cfg = config.ConstantConfig(constant_config_file)
scenario_cfgs = config.ScenarioConfig(scenario_file)

# Combine configurations
scenario_cfg = constant_cfg + scenario_cfgs[0]

# Initialize simulation
sim = load_simulation_environment(scenario_cfg)

# Run the simulation
sim.run()

--------------------------------------------------------------------------------
Simulation of scenario example_pool_irsonly_sc_1
Only minimum output to console -> see log-file
Scenario example_pool_irsonly_sc_1
Operator 0 control strategy: PoolingInsertionHeuristicOnly
	 control function: {'func_key': 'distance_and_user_times_with_walk', 'vot': 0.45}
Operator 0 additional strategies:
	 RV Heuristics: {}
	 Stop-Insert Heuristics: {}
	 Charging: None
	 On-Street Parking: True
	 Repositioning: None
	 Dynamic Pricing: None
	 Dynamic Fleet Sizing: None



example_pool_irsonly_sc_1: 100%|██████████| 100/100 [00:05<00:00, 17.51it/s, simulation_time=3570, driving=4, idle=6, charging=0, reposition=0]


Scenario example_pool_irsonly_sc_1 finished:
      initialization : 0:00:01 h
          simulation : 0:00:05 h
          evaluation : 0:00:00 h



## Analysis

In [4]:
# Define the results directory based on the scenario name
results_dir = os.path.join(os.getcwd(), 'results',
                           scenario_cfg[G_SCENARIO_NAME])

In [5]:
# Get KPI summary
kpi_summary = analysis.analyze_kpis(results_dir)

# Display KPIs
print('📈 Simulation KPIs:')
print(kpi_summary.to_string(index=False))

📈 Simulation KPIs:
                 Metric      Value
         Total Requests  48.000000
        Served Requests  48.000000
Average Wait Time (min)   1.927324
Average Trip Time (min)   3.074630
    Total Distance (km) 119.855174
    Empty Distance (km)  44.103579
       Service Rate (%) 100.000000
     Created Offers (%) 100.000000
  Fleet Utilization (%)  39.904432
     Occupancy Rate (%)   0.632026


In [6]:
# Analyze user statistics
fig, wait_time_stats = analysis.analyze_user_stats(results_dir)
fig.show()

print('\n⏱️ Wait Time Statistics (minutes):')
print(wait_time_stats.to_string())


⏱️ Wait Time Statistics (minutes):
count    0.800000
mean     0.032122
std      0.023121
min      0.000000
25%      0.010205
50%      0.030377
75%      0.049359
max      0.077168
